# Kapruka MCP — Tool Test Notebook

## 0 · Setup

In [1]:
import sys, os, json
from pathlib import Path

# Fix event loop conflict on Windows + Jupyter
import nest_asyncio
nest_asyncio.apply()

import asyncio

# Locate project root
project_root = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'kapruka_mcp').is_dir()),
    None
)
if project_root is None:
    raise RuntimeError('Could not find project root with kapruka_mcp folder')
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from dotenv import load_dotenv
load_dotenv(project_root / '.env')

# Pretty printer
def show(label, data):
    print('\n' + '='*55)
    print(f'  {label}')
    print('='*55)
    print(json.dumps(data, indent=2, default=str))

# Sync wrapper — avoids await conflicts in Jupyter on Windows
def run(coro):
    return asyncio.get_event_loop().run_until_complete(coro)

print(f'✅ Setup complete')
print(f'   Root : {project_root}')
print(f'   kapruka_mcp found: {(project_root / "kapruka_mcp").is_dir()}')

✅ Setup complete
   Root : c:\Projects\Agent\Kapruka_Agent_contest
   kapruka_mcp found: True


## 1 · Rate Limiter Status

In [2]:
from kapruka_mcp.rate_limiter import get_rate_limit_status

status = get_rate_limit_status()
show('Rate Limit Status', status)

assert status['requests_remaining_this_minute'] > 0, '❌ No requests remaining!'
print('✅ Rate limiter has capacity')


  Rate Limit Status
{
  "requests_used_this_minute": 0,
  "requests_remaining_this_minute": 60,
  "create_orders_used_this_hour": 0,
  "create_orders_remaining_this_hour": 30
}
✅ Rate limiter has capacity


## 2 · `kapruka_list_categories`

In [3]:
import httpx

async def probe():
    paths = ["/mcp", "/mcp/", "/sse", "/", ""]
    methods = ["GET", "POST"]
    async with httpx.AsyncClient(timeout=10) as client:
        for path in paths:
            for method in methods:
                try:
                    url = f"https://mcp.kapruka.com{path}"
                    if method == "POST":
                        r = await client.post(url,
                            json={"jsonrpc":"2.0","id":"1","method":"tools/list","params":{}},
                            headers={"Content-Type":"application/json","Accept":"application/json,text/event-stream"})
                    else:
                        r = await client.get(url)
                    print(f"{method} {path or '/'} → {r.status_code} | {r.headers.get('content-type','')}")
                except Exception as e:
                    print(f"{method} {path or '/'} → ERROR: {e}")

run(probe())

GET /mcp → 406 | application/json
POST /mcp → 400 | application/json
GET /mcp/ → 307 | 
POST /mcp/ → 307 | 
GET /sse → 404 | text/plain; charset=utf-8
POST /sse → 404 | text/plain; charset=utf-8
GET / → 200 | text/html; charset=utf-8
POST / → 405 | text/plain; charset=utf-8
GET / → 200 | text/html; charset=utf-8
POST / → 405 | text/plain; charset=utf-8


In [4]:
import httpx, asyncio, json

async def debug():
    async with httpx.AsyncClient(timeout=10, follow_redirects=True) as client:
        # Test 1: tools/call
        r1 = await client.post("https://mcp.kapruka.com/mcp",
            json={"jsonrpc":"2.0","id":"1","method":"tools/call",
                  "params":{"name":"kapruka_list_categories","arguments":{}}},
            headers={"Content-Type":"application/json","Accept":"application/json, text/event-stream"})
        print(f"tools/call → {r1.status_code}")
        print(r1.text[:500])
        print()

        # Test 2: tools/list (discover what tools exist)
        r2 = await client.post("https://mcp.kapruka.com/mcp",
            json={"jsonrpc":"2.0","id":"2","method":"tools/list","params":{}},
            headers={"Content-Type":"application/json","Accept":"application/json, text/event-stream"})
        print(f"tools/list → {r2.status_code}")
        print(r2.text[:500])
        print()

        # Test 3: initialize first (some servers require handshake)
        r3 = await client.post("https://mcp.kapruka.com/mcp",
            json={"jsonrpc":"2.0","id":"3","method":"initialize",
                  "params":{"protocolVersion":"2024-11-05",
                            "capabilities":{},"clientInfo":{"name":"test","version":"1.0"}}},
            headers={"Content-Type":"application/json","Accept":"application/json, text/event-stream"})
        print(f"initialize → {r3.status_code}")
        print(r3.text[:500])

run(debug())

tools/call → 400
{"jsonrpc":"2.0","id":"server-error","error":{"code":-32600,"message":"Bad Request: Missing session ID"}}

tools/list → 400
{"jsonrpc":"2.0","id":"server-error","error":{"code":-32600,"message":"Bad Request: Missing session ID"}}

initialize → 200
event: message
data: {"jsonrpc":"2.0","id":"3","result":{"protocolVersion":"2024-11-05","capabilities":{"experimental":{},"prompts":{"listChanged":false},"resources":{"subscribe":false,"listChanged":false},"tools":{"listChanged":false}},"serverInfo":{"name":"kapruka_mcp","version":"1.30.0"},"instructions":"You are connected to the Kapruka MCP server for Kapruka.com — Sri Lanka's largest e-commerce platform. Use the tools to search products, browse categories, look up product details, quote deli


In [5]:
from kapruka_mcp.mcp_client import MCP_URL, BASE
print(f"BASE = {BASE}")
print(f"MCP_URL = {MCP_URL}")

# Test initialize directly
import httpx, uuid

async def test_init():
    async with httpx.AsyncClient(timeout=30, follow_redirects=True) as client:
        r = await client.post(
            MCP_URL,
            json={"jsonrpc":"2.0","id":str(uuid.uuid4()),"method":"initialize",
                  "params":{"protocolVersion":"2024-11-05","capabilities":{},
                            "clientInfo":{"name":"test","version":"1.0"}}},
            headers={"Content-Type":"application/json",
                     "Accept":"application/json, text/event-stream"},
        )
        print(f"status: {r.status_code}")
        print(f"headers: {dict(r.headers)}")
        print(f"body: {r.text[:300]}")

run(test_init())

BASE = https://mcp.kapruka.com
MCP_URL = https://mcp.kapruka.com/mcp
status: 200
headers: {'date': 'Sun, 20 Sep 2026 05:34:54 GMT', 'content-type': 'text/event-stream', 'transfer-encoding': 'chunked', 'connection': 'keep-alive', 'alt-svc': 'h3=":443"; ma=86400', 'cache-control': 'no-cache, no-transform', 'mcp-session-id': '841af0a13b1f44568aee47f66b0c35e1', 'permissions-policy': 'geolocation=(), microphone=(), camera=()', 'ratelimit-limit': '60', 'ratelimit-remaining': '46', 'ratelimit-reset': '54', 'referrer-policy': 'strict-origin-when-cross-origin', 'strict-transport-security': 'max-age=15552000; preload', 'via': '1.1 Caddy', 'x-content-type-options': 'nosniff', 'cf-cache-status': 'DYNAMIC', 'server': 'cloudflare', 'cf-ray': 'a3de76b51d592664-CMB'}
body: event: message
data: {"jsonrpc":"2.0","id":"ed6568d1-cb82-47d5-9b1e-c96ea94dd11a","result":{"protocolVersion":"2024-11-05","capabilities":{"experimental":{},"prompts":{"listChanged":false},"resources":{"subscribe":false,"listChanged

In [6]:
from kapruka_mcp.tools.list_categories import list_categories

result = run(list_categories(depth=1))
show('list_categories (depth=1)', result)

assert result, '❌ Empty response'
print('✅ Got response')

[INFO] [McpClient] 2026-09-20T11:04:52 — MCP session initialized: eb5a317175944f03bc78fd6ec8d3a33f
[DEBUG] [McpClient] 2026-09-20T11:04:52 — → kapruka_list_categories | params={'params': {'depth': 1}}
[DEBUG] [McpClient] 2026-09-20T11:04:54 — ← kapruka_list_categories | status=200

  list_categories (depth=1)
{
  "text": "## Kapruka Categories\n\n- [Automobile](https://www.kapruka.com/online/automobile)\n- [Ayurvedic](https://www.kapruka.com/online/ayurvedic)\n- [Bicycles](https://www.kapruka.com/online/bicycles)\n- [Books](https://www.kapruka.com/online/books)\n- [Chocolates](https://www.kapruka.com/online/chocolates)\n- [Clothing](https://www.kapruka.com/online/clothing)\n- [combopack](https://www.kapruka.com/online/combogifts)\n- [Cosmetics](https://www.kapruka.com/online/cosmetics)\n- [Curd](https://www.kapruka.com/online/curd)\n- [Electronic](https://www.kapruka.com/online/electronics)\n- [Fashion](https://www.kapruka.com/online/fashion)\n- [Fruits](https://www.kapruka.com/online/

## 3 · `kapruka_search_products` — keyword

In [7]:
from kapruka_mcp.tools.search_products import search_products

result = run(search_products(
    q='birthday cake',
    in_stock_only=True,
    limit=3,
    currency='LKR'
))
show('search_products (birthday cake)', result)

assert result, '❌ Empty response'
print('✅ Search returned results')

[DEBUG] [McpClient] 2026-09-20T11:04:54 — → kapruka_search_products | params={'params': {'q': 'birthday cake', 'in_stock_only': True, 'limit': 3, 'currency': 'LKR'}}
[DEBUG] [McpClient] 2026-09-20T11:04:55 — ← kapruka_search_products | status=200

  search_products (birthday cake)
{
  "text": "No products found for 'birthday cake'."
}
✅ Search returned results


## 4 · `kapruka_search_products` — price range

In [8]:
result = run(search_products(
    q='flowers',
    min_price=1000,
    max_price=5000,
    sort='price_asc',
    limit=3,
    currency='LKR'
))
show('search_products (flowers LKR 1000-5000)', result)
print('✅ Price filter search done')

[DEBUG] [McpClient] 2026-09-20T11:04:55 — → kapruka_search_products | params={'params': {'q': 'flowers', 'min_price': 1000, 'max_price': 5000, 'sort': 'price_asc', 'limit': 3, 'currency': 'LKR'}}
[DEBUG] [McpClient] 2026-09-20T11:04:56 — ← kapruka_search_products | status=200

  search_products (flowers LKR 1000-5000)
{
  "text": "## Kapruka search: \"flowers\"\nShowing 3 results (LKR)\n\n**1. Bath Scoop Blue**\n   ID: `EF_PC_MOTH0V2686POD00178` \u00b7 LKR 1,000 \u00b7 In stock (low) \u00b7 ships internationally\n   [View product](https://www.kapruka.com/buyonline/bath-scoop-blue/kid/ef_pc_moth0v2686pod00178)\n\n**2. Kapruka Gift Voucher**\n   ID: `GIFTV00Z235` \u00b7 LKR 1,000 \u00b7 In stock (low) \u00b7 ships internationally\n   [View product](https://www.kapruka.com/buyonline/kapruka-gift-voucher/kid/giftv00z235)\n\n**3. Daisy Candle Greeting Card - Best Mother Ever, Sri Lanka**\n   ID: `EF_PC_GREE0V44P00284` \u00b7 LKR 1,000 \u00b7 In stock (low) \u00b7 ships internationally\n   [

## 5 · `kapruka_get_product`
Copy a `product_id` from cell 3 or 4 results and paste below.

In [16]:
from kapruka_mcp.tools.get_product import get_product

PRODUCT_ID = 'EF_PC_MOTH0V2686POD00178'  # ← paste here

result = run(get_product(product_id=PRODUCT_ID, currency='LKR'))
show(f'get_product ({PRODUCT_ID})', result)

assert result, '❌ Empty response'
print('✅ Product details retrieved')
print('Fields:', list(result.keys()) if isinstance(result, dict) else 'see above')

[DEBUG] [McpClient] 2026-09-20T11:07:15 — → kapruka_get_product | params={'params': {'product_id': 'EF_PC_MOTH0V2686POD00178', 'currency': 'LKR'}}
[DEBUG] [McpClient] 2026-09-20T11:07:16 — ← kapruka_get_product | status=200

  get_product (EF_PC_MOTH0V2686POD00178)
{
  "text": "## Bath Scoop Blue\n**ID**: `EF_PC_MOTH0V2686POD00178`\n**Price**: LKR 1,000\n**Stock**: In stock (high)\n**Category**: MOTHER AND BABY\n**Vendor**: kidsmarket\n**Weight**: 0 lbs\n**International shipping**: Yes\n**Delivery**: Island-wide\n\nThe Bath Scoop Blue is a must-have  bath and hygiene  tool for your  baby  s bath time, available at Kapruka in Sri Lanka. It s designed to make rinsing easy and fun, ensuring a smooth bathing expe...\n\n**Image**: https://partnercentral.kapruka.com/kapruka-pc/assets/images/product/pc01192/moth0v2686p00178/moth0v2686p00178_1.jpg\n\n[View on Kapruka](https://www.kapruka.com/buyonline/bath-scoop-blue/kid/ef_pc_moth0v2686pod00178)"
}
✅ Product details retrieved
Fields: ['text']

## 6 · `kapruka_list_delivery_cities`

In [10]:
from kapruka_mcp.tools.list_delivery_cities import list_delivery_cities

result_colombo = run(list_delivery_cities(query='Colombo'))
show('list_delivery_cities (Colombo)', result_colombo)

result_kandy = run(list_delivery_cities(query='Kandy'))
show('list_delivery_cities (Kandy)', result_kandy)

print('✅ City lookup done')

[DEBUG] [McpClient] 2026-09-20T11:04:57 — → kapruka_list_delivery_cities | params={'params': {'query': 'Colombo'}}
[DEBUG] [McpClient] 2026-09-20T11:04:58 — ← kapruka_list_delivery_cities | status=200

  list_delivery_cities (Colombo)
{
  "text": "## Kapruka delivery cities \u2014 'Colombo' (15 of 15)\n\n- **Colombo 01**  _aliases: Colombo1_\n- **Colombo 02**  _aliases: Slave Colombo2_\n- **Colombo 03**  _aliases: Kolpity colpity colombo3_\n- **Colombo 04**  _aliases: bambala colombo4_\n- **Colombo 05**  _aliases: thimbirigasyaya kirulapona narahenpita thibirigas_\n- **Colombo 06**  _aliases: wellawatta walawtha wellawatha colombo6 welawathth_\n- **Colombo 07**  _aliases: Colombo7_\n- **Colombo 08**  _aliases: borella boralla colombo8_\n- **Colombo 09**  _aliases: Colombo9 dematagoda_\n- **Colombo 10**  _aliases: maradana_\n- **Colombo 11**  _aliases: peta_\n- **Colombo 12**\n- **Colombo 13**  _aliases: Kotahena_\n- **Colombo 14**  _aliases: grandpass_\n- **Colombo 15**  _aliases: mata

## 7 · `kapruka_check_delivery` — no product_id

In [11]:
from kapruka_mcp.tools.check_delivery import check_delivery
from datetime import date, timedelta

tomorrow = (date.today() + timedelta(days=1)).isoformat()
print(f'Delivery date: {tomorrow}')

result = run(check_delivery(city='Colombo', delivery_date=tomorrow))
show(f'check_delivery (Colombo, {tomorrow})', result)

print('✅ Delivery check done')

Delivery date: 2026-09-21
[DEBUG] [McpClient] 2026-09-20T11:04:59 — → kapruka_check_delivery | params={'params': {'city': 'Colombo', 'delivery_date': '2026-09-21'}}
[DEBUG] [McpClient] 2026-09-20T11:05:00 — ← kapruka_check_delivery | status=200

  check_delivery (Colombo, 2026-09-21)
{
  "text": "Error (city_not_found): Unknown city 'Colombo'"
}
✅ Delivery check done


## 8 · `kapruka_check_delivery` — with product_id

In [12]:
result = run(check_delivery(
    city='Kandy',
    delivery_date=tomorrow,
    product_id=PRODUCT_ID
))
show(f'check_delivery (Kandy + product)', result)
print('✅ Product-specific delivery check done')

[DEBUG] [McpClient] 2026-09-20T11:05:00 — → kapruka_check_delivery | params={'params': {'city': 'Kandy', 'delivery_date': '2026-09-21', 'product_id': 'REPLACE_WITH_PRODUCT_ID'}}
[DEBUG] [McpClient] 2026-09-20T11:05:01 — ← kapruka_check_delivery | status=200

  check_delivery (Kandy + product)
{
  "text": "## Delivery to Kandy on 2026-09-21\n**Available** \u2014 flat rate LKR 1,075"
}
✅ Product-specific delivery check done


## 9 · `kapruka_create_order` — DRY RUN
⚠️ Consumes 1 of your 30 orders/hr cap. Blocked by default.

In [13]:
from kapruka_mcp.tools.create_order import create_order

RUN_ORDER_TEST = False  # ← set True to run

if not RUN_ORDER_TEST:
    print('⏭️  Skipped — set RUN_ORDER_TEST = True to run')
else:
    result = run(create_order(
        cart=[{'product_id': PRODUCT_ID, 'quantity': 1}],
        recipient={'name': 'Test Recipient', 'phone': '0771234567',
                   'address': '123 Test Street', 'city': 'Colombo'},
        delivery={'date': tomorrow},
        sender={'name': 'Test Sender', 'phone': '0777654321'},
        gift_message='Test order from notebook',
        currency='LKR'
    ))
    show('create_order result', result)
    print('✅ Order created — payment URL valid for 60 min')

⏭️  Skipped — set RUN_ORDER_TEST = True to run


## 10 · `kapruka_track_order`

In [14]:
from kapruka_mcp.tools.track_order import track_order

ORDER_NUMBER = 'REPLACE_WITH_ORDER_NUMBER'  # ← paste here

result = run(track_order(order_number=ORDER_NUMBER))
show(f'track_order ({ORDER_NUMBER})', result)
print('✅ Tracking done')

[DEBUG] [McpClient] 2026-09-20T11:05:01 — → kapruka_track_order | params={'params': {'order_number': 'REPLACE_WITH_ORDER_NUMBER'}}
[DEBUG] [McpClient] 2026-09-20T11:05:02 — ← kapruka_track_order | status=200

  track_order (REPLACE_WITH_ORDER_NUMBER)
{
  "text": "Error (order_not_found): No order exists with the given order number"
}
✅ Tracking done


## 11 · Final rate limit status

In [15]:
status = get_rate_limit_status()
show('Rate Limit Status (after tests)', status)
print(f"Used {status['requests_used_this_minute']} requests, {status['requests_remaining_this_minute']} remaining")


  Rate Limit Status (after tests)
{
  "requests_used_this_minute": 9,
  "requests_remaining_this_minute": 51,
  "create_orders_used_this_hour": 0,
  "create_orders_remaining_this_hour": 30
}
Used 9 requests, 51 remaining
